# toc

> Generate structured Table of Contents from video xscripts via LLM

In [ ]:
#| default_exp toc

## Design

LLM returns `[{title, start}, ...]`. App-side adds `path` (sequential) and `end` (next start or duration).

**Pipeline:**
1. `_build_toc_prompt(segments, meta)` — build prompt from xscript segments
2. `_call_llm(prompt)` — call OpenAI gpt-5.4, return raw `[{title, start}]`
3. `_normalize_sections(raw, duration)` — add path/end, sort, dedup, validate
4. Write `toc.json` to cache dir

**toc.json format:**
```json
{"sections": [{"path": "1", "title": "Introduction", "start": 0, "end": 750}]}
```

In [ ]:
#| export
import json
from pathlib import Path
from pydantic import BaseModel, Field
from yttoc.core import Segment, NormalizedSection, Meta
from yttoc.llm import generate_structured

In [ ]:
#| export
def _normalize_sections(raw: 'list[RawTocSection]', # [RawTocSection, ...] from LLM
                        duration: int # Video duration in seconds
                       ) -> list[NormalizedSection]: # List of NormalizedSection
    "Add path/end, sort by start, dedup, validate. Raise on broken coverage."
    if not raw:
        raise ValueError("No sections returned from LLM")
    # Sort by start ascending
    sections = sorted(raw, key=lambda s: s.start)
    # Remove duplicate starts (keep first)
    seen = set()
    deduped = []
    for s in sections:
        if s.start not in seen:
            seen.add(s.start)
            deduped.append(s)
    sections = deduped
    # Fix first section start to 0 (create a new RawTocSection with start=0)
    sections[0] = sections[0].model_copy(update={'start': 0})
    # Build NormalizedSection list
    result = []
    for i, s in enumerate(sections):
        end = sections[i+1].start if i+1 < len(sections) else duration
        result.append(NormalizedSection(path=str(i+1), title=s.title, start=s.start, end=end))
    return result


In [ ]:
#| export
def _build_toc_prompt(segments: list[Segment], # List of Segment from parse_xscript
                      meta: Meta # Parsed Meta instance
                     ) -> str: # Prompt for LLM
    "Build a prompt that asks the LLM to identify topic transitions and return section titles with start times."
    lines = []
    for s in segments:
        mm = int(s.start // 60)
        ss = int(s.start % 60)
        lines.append(f"[{mm:02d}:{ss:02d}] {s.text}")
    transcript = '\n'.join(lines)

    title = meta.title
    channel = meta.channel
    desc = meta.description

    return f"""You are a structural editor for YouTube video transcripts.

Video info:
- Title: {title}
- Channel: {channel}
- Description: {desc}

Transcript:
{transcript}

Task:
Read the transcript and identify topic transitions. For each section, provide:
- title: concise English section title
- start: start time in integer seconds

Aim for 7-15 sections. Be faithful to transcript timestamps."""

class RawTocSection(BaseModel):
    "One section as returned by the TOC LLM — title + start time only."
    title: str = Field(description="Concise English section title")
    start: int = Field(ge=0, description="Start time in integer seconds")

class TocLLMResult(BaseModel):
    "Structured output from the TOC generation LLM call."
    sections: list[RawTocSection]


class TocFile(BaseModel):
    "On-disk shape of toc.json."
    sections: list[NormalizedSection]

def _call_llm(prompt: str # Full prompt
             ) -> list[RawTocSection]: # List of RawTocSection
    "Call OpenAI gpt-5.4 with Pydantic-generated schema, return raw section list."
    result = generate_structured(prompt, TocLLMResult, schema_name='generate_toc')
    return result.sections


## Tests

In [ ]:
# Test 1: basic — adds path and computes end from next start; last end = duration
from yttoc.toc import RawTocSection
from yttoc.core import NormalizedSection
raw = [RawTocSection(title='Intro', start=0),
       RawTocSection(title='Main', start=300),
       RawTocSection(title='Outro', start=600)]
secs = _normalize_sections(raw, duration=900)
assert len(secs) == 3
assert secs[0] == NormalizedSection(path='1', title='Intro', start=0, end=300)
assert secs[1] == NormalizedSection(path='2', title='Main', start=300, end=600)
assert secs[2] == NormalizedSection(path='3', title='Outro', start=600, end=900)
print('ok')


In [ ]:
# Test 2: sorts by start ascending
from yttoc.toc import RawTocSection
raw = [RawTocSection(title='B', start=300), RawTocSection(title='A', start=0)]
secs = _normalize_sections(raw, duration=600)
assert secs[0].title == 'A'
assert secs[1].title == 'B'
print('ok')


In [ ]:
# Test 3: removes duplicate starts (keeps first occurrence after sort)
from yttoc.toc import RawTocSection
raw = [RawTocSection(title='A', start=0),
       RawTocSection(title='A-dup', start=0),
       RawTocSection(title='B', start=300)]
secs = _normalize_sections(raw, duration=600)
assert len(secs) == 2
assert secs[0].title == 'A'
assert secs[1].title == 'B'
print('ok')


In [ ]:
# Test 4: fixes first section start to 0 if not already
from yttoc.toc import RawTocSection
raw = [RawTocSection(title='Late start', start=30), RawTocSection(title='Next', start=300)]
secs = _normalize_sections(raw, duration=600)
assert secs[0].start == 0
assert secs[0].end == 300
print('ok')


In [ ]:
# Test 5: empty input raises ValueError
try:
    _normalize_sections([], duration=600)
    assert False, 'should have raised'
except ValueError:
    pass
print('ok')

In [ ]:
# Test: TocFile validates envelope shape and element types
from yttoc.toc import TocFile
from pydantic import ValidationError

# Valid
toc = TocFile.model_validate_json(
    '{"sections": [{"path":"1","title":"Intro","start":0,"end":300}]}'
)
assert len(toc.sections) == 1
assert toc.sections[0].title == 'Intro'

# Missing 'sections' key rejected
try:
    TocFile.model_validate_json('{"section": []}')  # typo
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing sections key'

# Bad element shape (missing 'end') rejected
try:
    TocFile.model_validate_json(
        '{"sections": [{"path":"1","title":"x","start":0}]}'
    )
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing end field'

# Negative timestamp rejected via NormalizedSection constraint
try:
    TocFile.model_validate_json(
        '{"sections": [{"path":"1","title":"x","start":-1,"end":10}]}'
    )
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative start'

print('ok')

In [ ]:
from yttoc.core import Segment, Meta
from datetime import datetime, timezone
# Test 6: _build_toc_prompt includes transcript and meta context
segments = [
    Segment(start=0.0, end=5.0, text='hello world'),
    Segment(start=5.0, end=10.0, text='second segment'),
]
meta = Meta(
    id='T', title='Test Video', channel='Ch',
    duration=600, upload_date='20260101',
    webpage_url='https://youtube.com/watch?v=T',
    description='A test video', captions={'en': 'auto'},
    last_used_at=datetime(2026, 1, 1, tzinfo=timezone.utc))
prompt = _build_toc_prompt(segments, meta)
assert 'hello world' in prompt
assert 'second segment' in prompt
assert 'Test Video' in prompt
assert '[00:00]' in prompt  # timestamps formatted
print('ok')

## CLI

In [ ]:
#| export
import sys
from fastcore.script import call_parse
from yttoc.core import format_header, format_toc_line, Meta
from yttoc.cache import (resolve_root, meta_path, toc_path, summaries_path,
                         first_srt_path, load_meta, read_model)
from yttoc.fetch import _update_last_used
from yttoc.xscript import parse_xscript

def generate_toc(video_id: str, # Exact video_id
                 root: Path = None, # Root cache directory
                 refresh: bool = False, # Delete cached toc/summaries and regenerate
                ) -> list[NormalizedSection]: # List of NormalizedSection
    "Generate toc.json for a cached video. Returns sections list."
    root = resolve_root(root)
    meta_p = meta_path(video_id, root)
    toc_p = toc_path(video_id, root)
    sum_p = summaries_path(video_id, root)
    if not meta_p.exists():
        raise SystemExit(f"Not cached: {video_id}")
    try:
        srt_path = first_srt_path(video_id, root)
    except FileNotFoundError:
        raise SystemExit(f"Not cached: {video_id}")

    if refresh:
        if toc_p.exists(): toc_p.unlink()
        if sum_p.exists():
            sum_p.unlink()
            print('Invalidated summaries.json (depends on toc)', file=sys.stderr)

    if toc_p.exists():
        return read_model(toc_p, TocFile).sections

    meta = load_meta(video_id, root)
    segments = parse_xscript(srt_path)
    prompt = _build_toc_prompt(segments, meta)
    raw = _call_llm(prompt)
    sections = _normalize_sections(raw, meta.duration)

    toc_p.write_text(
        TocFile(sections=sections).model_dump_json(indent=2),
        encoding='utf-8')
    _update_last_used(meta_p)
    return sections

def _render_toc(meta: Meta, # Parsed Meta instance
                sections: list[NormalizedSection], # Normalized TOC sections
               ) -> str: # Rendered TOC: header + blank + formatted lines
    "Render TOC output for yttoc_toc."
    lines = [format_header(meta), '']
    url = meta.webpage_url
    for s in sections:
        lines.append(format_toc_line(s, url))
    return '\n'.join(lines)

@call_parse
def yttoc_toc(video_id: str, # Exact video_id
              root: str = None, # Root cache directory
              refresh: bool = False, # Regenerate toc (and invalidate summaries)
             ):
    "Generate and display Table of Contents for a cached video."
    root = resolve_root(root)
    meta_p = meta_path(video_id, root)
    if not meta_p.exists():
        raise SystemExit(f"Not cached: {video_id}")

    meta = load_meta(video_id, root)
    sections = generate_toc(video_id, root, refresh=refresh)
    print(_render_toc(meta, sections))


In [ ]:
# Test 7: generate_toc returns cached toc.json without calling LLM
from tempfile import TemporaryDirectory
import io, contextlib

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID1',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00'}))
    # Pre-write toc.json so LLM is never called
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))

    secs = generate_toc('VID1', root)
    assert len(secs) == 2
    assert secs[0].title == 'Intro'
    assert secs[1].title == 'Main'
print('ok')


In [ ]:
# Test 8: yttoc_toc outputs header + formatted section lines
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 'Test Video', 'channel': 'Ch', 'duration': 900,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID2',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 900},
    ]}))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_toc('VID2', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert '1. Intro 0:00-5:00' in out
    assert '2. Main 5:00-15:00' in out
    assert '(5:00)' in out  # span duration
    assert '&t=300' in out  # deep link
print('ok')

In [ ]:
# Test 9: TocFile rejects a corrupted toc.json (negative start)
from tempfile import TemporaryDirectory
from pydantic import ValidationError

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_BAD'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID_BAD', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID_BAD',
        'last_used_at': '2000-01-01T00:00:00'}))
    # Corrupted toc.json — negative start violates NormalizedSection(start ≥ 0)
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': -1, 'end': 300},
    ]}))

    try:
        generate_toc('VID_BAD', root)
    except ValidationError:
        pass
    else:
        assert False, 'expected ValidationError for negative start in toc.json'
print('ok')

In [ ]:
# Test 10: generate_toc cache-hit returns list[NormalizedSection] with typed fields
from tempfile import TemporaryDirectory
from yttoc.core import NormalizedSection

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_OK'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID_OK', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID_OK',
        'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))

    secs = generate_toc('VID_OK', root)
    assert len(secs) == 2
    assert all(isinstance(s, NormalizedSection) for s in secs)
    assert secs[0].path == '1' and secs[0].title == 'Intro' and secs[0].start == 0 and secs[0].end == 300
    assert secs[1].path == '2' and secs[1].title == 'Main' and secs[1].start == 300 and secs[1].end == 600
print('ok')

In [ ]:
# Test: _render_toc returns header + blank + formatted section lines
from yttoc.toc import _render_toc
from yttoc.core import NormalizedSection, Meta
from datetime import datetime, timezone

meta = Meta(id='VID2', title='Test Video', channel='Ch', duration=900,
            upload_date='20260101', webpage_url='https://youtube.com/watch?v=VID2',
            description='', captions={'en': 'auto'},
            last_used_at=datetime(2000,1,1,tzinfo=timezone.utc))
sections = [NormalizedSection(path='1', title='Intro', start=0, end=300),
            NormalizedSection(path='2', title='Main', start=300, end=900)]

out = _render_toc(meta, sections)
lines = out.split('\n')
assert '# Test Video' in lines[0]
assert lines[2] == ''  # blank line between header (2 lines) and first section
assert '1. Intro 0:00-5:00' in out
assert '2. Main 5:00-15:00' in out
assert '&t=300' in out
print('ok')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()